# Ingestion — micro-batch data ingestion

Simulates the daily arrival of a new CSV drop and appends it to the running `training_table.csv`. Validates schema on every ingest and skips a file it has already ingested (detected via content hash), so re-running this notebook on the same day's file is safe.

Set `INPUT_FILE` in the cell below to the file you want to ingest, then run all cells.

In [1]:
import hashlib, json, os
from datetime import datetime, timezone
import pandas as pd

TABLE_PATH = '../data/processed/training_table.csv'
LOG_PATH = '../data/processed/ingestion_log.jsonl'

REQUIRED_COLUMNS = [
    'customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
    'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn',
]


## Parameter — which file to ingest

Edit this and re-run for each new daily drop.

In [2]:
INPUT_FILE = '../data/raw/daily_2026_08_07.csv'  # <- change per run

## Ingestion function

In [3]:
def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        h.update(f.read())
    return h.hexdigest()[:16]

def log_ingestion(record):
    os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)
    with open(LOG_PATH, 'a') as f:
        f.write(json.dumps(record) + '\n')

def ingest(input_path, table_path=TABLE_PATH, log_path=LOG_PATH):
    if not os.path.exists(input_path):
        raise FileNotFoundError(input_path)

    new_df = pd.read_csv(input_path)
    missing = [c for c in REQUIRED_COLUMNS if c not in new_df.columns]
    if missing:
        raise ValueError(f'Ingestion schema check failed. Missing columns: {missing}')

    h = file_hash(input_path)
    already = False
    if os.path.exists(log_path):
        with open(log_path) as f:
            for line in f:
                if json.loads(line).get('file_hash') == h:
                    already = True
                    break

    if already:
        print(f'[ingest] SKIP: {input_path} (hash {h}) already ingested.')
        return {'status': 'skipped', 'rows': 0}

    header_needed = not os.path.exists(table_path)
    os.makedirs(os.path.dirname(table_path), exist_ok=True)
    new_df.to_csv(table_path, mode='a', header=header_needed, index=False)

    record = {
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
        'source_file': input_path, 'file_hash': h,
        'rows_ingested': len(new_df), 'target_table': table_path,
    }
    log_ingestion(record)
    print(f'[ingest] OK: appended {len(new_df)} rows from {input_path} -> {table_path}')
    return {'status': 'ok', 'rows': len(new_df)}


## Run

In [4]:
result = ingest(INPUT_FILE)
result

[ingest] OK: appended 500 rows from ../data/raw/daily_2026_08_07.csv -> ../data/processed/training_table.csv


{'status': 'ok', 'rows': 500}

## Verify table state

In [5]:
if os.path.exists(TABLE_PATH):
    df = pd.read_csv(TABLE_PATH)
    print(f'Training table now has {len(df)} rows.')
    display(df.tail(3))


Training table now has 6500 rows.


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
6497,0815-MFZGM,Female,0,Yes,No,42,Yes,No,Fiber optic,No,...,Yes,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),99.00,4135,No
6498,5774-XZTQC,Female,0,Yes,Yes,7,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.45,150.75,No
6499,2916-BQZLN,Male,0,No,No,19,Yes,Yes,Fiber optic,Yes,...,Yes,No,No,No,Month-to-month,Yes,Electronic check,84.75,1651.95,No
